In [1]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    Seq2SeqTrainer, 
    Seq2SeqTrainingArguments, 
    DataCollatorForSeq2Seq
)

# 1. SETUP & CONFIGURATION
# ---------------------------------------------------------
model_checkpoint = "allenai/led-base-16384"
dataset_name = "datht/FINDSum" 

# Kaggle T4 GPU constraints
MAX_INPUT_LENGTH = 8192  # Reduced from 16k to fit in T4 memory
MAX_TARGET_LENGTH = 512
BATCH_SIZE = 1           # Keep low to avoid Out of Memory (OOM)
GRADIENT_ACCUMULATION = 4 # Simulates a batch size of 4

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

# 2. PREPARE THE DATASET
# ---------------------------------------------------------
# We load only 2000 examples for a demo run. 
# If you want to train for real, remove "split='train[:2000]'" 
# but be prepared for it to run for 20+ hours.
print("Loading dataset...")
dataset = load_dataset(dataset_name, split="train[:6000]") 
# Note: We need to split this into train/val manually since we sliced it
dataset = dataset.train_test_split(test_size=0.1)

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    # The dataset likely uses "text" and "summary" keys. 
    # If the code crashes here, print(examples.keys()) to check column names.
    inputs = examples["document"] 
    targets = examples["summary"]
    
    model_inputs = tokenizer(
        inputs, 
        max_length=MAX_INPUT_LENGTH, 
        padding="max_length", 
        truncation=True
    )

    # Setup the tokenizer for targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets, 
            max_length=MAX_TARGET_LENGTH, 
            padding="max_length", 
            truncation=True
        )

    # Replace padding token id with -100 so we don't calculate loss on padding
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing data (this may take a moment)...")
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# 3. LOAD MODEL
# ---------------------------------------------------------
print("Loading Model...")
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# Critical for LED: Set specific configuration for generation
model.config.num_beams = 2
model.config.max_length = 512
model.config.min_length = 100
model.config.length_penalty = 2.0
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3

# IMPORTANT: Enable Gradient Checkpointing to save memory
model.gradient_checkpointing_enable()

# 4. TRAIN
# ---------------------------------------------------------
training_args = Seq2SeqTrainingArguments(
    output_dir="./led-financial-finetuned",
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=1,          # Keep to 1 epoch for testing
    predict_with_generate=True,
    fp16=True,                   # CRITICAL: Uses mixed precision (requires GPU)
    logging_steps=50,
    report_to="none",            # Disable wandb to keep it simple
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

print("Starting Training...")
trainer.train()

# 5. SAVE
# ---------------------------------------------------------
print("Saving custom Financial LED...")
model.save_pretrained("./final_financial_led")
tokenizer.save_pretrained("./final_financial_led")
print("Done! You now have a Financial LED model.")

2025-12-01 10:16:30.933267: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764584191.115541      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764584191.167318      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

model.safetensors:   0%|          | 0.00/648M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss,Validation Loss
200,2.325800,2.195485
400,2.226500,2.118802
600,2.207200,2.083516


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3685: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'min_length': 100, 'early_stopping': True, 'num_beams': 2, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather al

Saving custom Financial LED...
Done! You now have a Financial LED model.
